# Setup & Configuration

In [52]:
# ─── Standard Library ───
import os
import re
import json
import time
import base64
from io import BytesIO
from pathlib import Path
import csv
import ast
import numpy as np

# ─── Third-Party ───
import requests
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from PIL import Image
import Levenshtein

### Configurations

In [ ]:
# ─── Retry budget for every API call ───
MAX_RETRIES = 3

# ─── OpenRouter credentials ───
# ─── Load credentials from .env ───
def load_env_key(key_name, default=None):
    env_path = Path(".env")
    if env_path.exists():
        with open(env_path, "r") as f:
            for line in f:
                if line.strip().startswith(f"{key_name}="):
                    return line.split("=", 1)[1].strip().strip('"').strip("'")
    return os.environ.get(key_name, default)

OPENROUTER_API_KEY = load_env_key("OPENROUTER_API_KEY")
# OPENROUTER_API_KEY = load_env_key("OPENROUTER_API_KEY_test") ## Test Key

# MODEL_NAME = "qwen/qwen3-vl-8b-instruct" #Done
# MODEL_NAME = "openai/gpt-4o-mini" #Done
# MODEL_NAME = "rekaai/reka-edge" # Done , remove 23696* 260 = 6,160,960 => $ 0.616096
# MODEL_NAME = "mistralai/ministral-8b-2512" #Done
# MODEL_NAME = "google/gemini-3.1-flash-lite-preview" #Done
# MODEL_NAME = "meta-llama/llama-3.2-11b-vision-instruct" #if needed
       

# ─── Dataset Paths ───
CHARTQA_PATH       = Path("../Datasets/chartqa.parquet")
AR_CHARTQA_PATH    = Path("../Datasets/chartqa_arabic.parquet")
MATHVISION_PATH    = Path("../Datasets/mathvision.parquet")
AR_MATHVISION_PATH = Path("../Datasets/mathvision_arabic.parquet")
SCREENQA_PATH      = Path("../Datasets/screenqa.parquet")
TURTLEBENCH_PATH   = Path("../Datasets/TurtleBench")
PEARL_PATH         = Path("../Datasets/pearl.parquet")
JEEM_PATH          = Path("../Datasets/jeem.parquet")

### Helper Functions

In [54]:
def encode_image_bytes_to_base64(byte_data, resize=None):
    """Convert raw image bytes to a base64 data-URL string.

    Parameters
    ----------
    byte_data : bytes
        Raw image bytes (e.g. from a parquet column).
    resize : tuple[int, int] | None
        Optional (width, height) cap; the image is thumbnailed to fit.

    Returns
    -------
    str | None
        A 'data:image/jpeg;base64,...' string, or None on failure.
    """
    try:
        img = Image.open(BytesIO(byte_data))
        if resize:
            img.thumbnail(resize)
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=85)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None


def encode_raw_bytes_to_base64(byte_data):
    """Encode raw bytes directly to a base64 data-URL (no PIL processing)."""
    try:
        b64 = base64.b64encode(byte_data).decode("utf-8")
        return f"data:image/jpeg;base64,{b64}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None


def encode_image_file_to_base64(image_path, include_prefix=True):
    """Open an image file from disk, convert to JPEG base64.

    Parameters
    ----------
    image_path : str | Path
        Path to the image on disk.
    include_prefix : bool
        If True, return with 'data:image/jpeg;base64,' prefix.

    Returns
    -------
    str | None
    """
    try:
        with Image.open(image_path) as img:
            if img.mode != "RGB":
                img = img.convert("RGB")
            buf = BytesIO()
            img.save(buf, format="JPEG", quality=85)
            b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
            return f"data:image/jpeg;base64,{b64}" if include_prefix else b64
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None


def query_openrouter(prompt, base64_image, model_name=None, max_retries=MAX_RETRIES):
    """Send an image + prompt to OpenRouter and return (response_text, duration)."""
    model = model_name or MODEL_NAME
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    # ── OpenAI vision token billing ────────────────────────────────────
    # OpenAI applies a ~33× billing multiplier to gpt-4o-mini image
    # tokens. Using detail:"low" forces a fixed 2833 billing tokens per
    # image instead of ~25 000 (detail:"high" or "auto").
    # This does NOT degrade quality for chart/text images.
    if model == "openai/gpt-4o-mini":
        image_block = {
            "type": "image_url",
            "image_url": {"url": base64_image, "detail": "low"},
        }
    else:
        image_block = {
            "type": "image_url",
            "image_url": {"url": base64_image},
        }

    payload = {
        "model": model,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                image_block,
            ],
        }],
        "temperature": 0.1,
    }

    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers, json=payload, timeout=120,
            )
            # This line triggers an HTTPError if the status code is 4xx or 5xx
            resp.raise_for_status() 
            
            text = resp.json()["choices"][0]["message"]["content"].strip()
            return text, time.time() - start
            
        except requests.exceptions.Timeout:
            wait_time = 5 * (attempt + 1)
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: timed out.")
            
        except requests.exceptions.RequestException as e:
            # Check specifically if the error is a 429 Rate Limit
            if getattr(e.response, 'status_code', None) == 429:
                # Apply a heavier backoff penalty specifically for rate limits
                wait_time = 10 * (attempt + 1) 
                print(f"   🛑 Rate limited (429)! Backing off for {wait_time}s...")
            else:
                wait_time = 5 * (attempt + 1)
                print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
                
        # Sleep before trying again
        if attempt < max_retries - 1:
            time.sleep(wait_time)

    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def query_openrouter_text(prompt, model_name=None, max_retries=MAX_RETRIES):
    """Send a text-only prompt to OpenRouter and return (response_text, duration)."""
    model = model_name or MODEL_NAME
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
    }

    start = time.time()
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers, json=payload, timeout=120,
            )
            resp.raise_for_status()
            text = resp.json()["choices"][0]["message"]["content"].strip()
            return text, time.time() - start
        except requests.exceptions.Timeout:
            wait_time = 5 * (attempt + 1)
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: timed out.")
        except requests.exceptions.RequestException as e:
            if getattr(e.response, 'status_code', None) == 429:
                wait_time = 10 * (attempt + 1)
                print(f"   🛑 Rate limited (429)! Backing off for {wait_time}s...")
            else:
                wait_time = 5 * (attempt + 1)
                print(f"   ❌ Attempt {attempt + 1}/{max_retries}: {e}")
        if attempt < max_retries - 1:
            time.sleep(wait_time)
    return "ERROR_MAX_RETRIES_REACHED", time.time() - start


def load_progress(output_csv, key_columns):
    """Load already-processed task keys from an existing CSV to enable resuming.

    Parameters
    ----------
    output_csv : Path
        CSV file path.
    key_columns : list[str]
        Column(s) whose concatenation forms a unique task key.

    Returns
    -------
    tuple[set, bool]
        (processed_keys, file_exists)
    """
    processed = set()
    exists = output_csv.is_file()
    if exists:
        try:
            df = pd.read_csv(output_csv, usecols=lambda c: c in key_columns, dtype=str, encoding="utf-8-sig")
            if not df.empty and all(c in df.columns for c in key_columns):
                if len(key_columns) == 1:
                    processed = set(df[key_columns[0]])
                else:
                    processed = set(
                        df[key_columns[0]].str.cat(
                            [df[c] for c in key_columns[1:]], sep="_"
                        )
                    )
                print(f"🔄 Resuming — skipping {len(processed)} already-processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV: {e}")
    return processed, exists


def append_result(result_dict, output_csv, file_exists):
    """Append a single result row to CSV, writing the header only once.

    Returns
    -------
    bool
        Updated file_exists flag (always True after first write).
    """
    # Use utf-8-sig (adds BOM) for new files so Excel recognises Arabic;
    # plain utf-8 for appends to avoid extra BOMs mid-file.
    enc = "utf-8-sig" if not file_exists else "utf-8"
    with open(output_csv, "a", newline="", encoding=enc) as f:
        writer = csv.DictWriter(f, fieldnames=result_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(result_dict)
    return True


def safe_model_filename(name):
    """Sanitise a model name for use inside a filename."""
    return name.replace("/", "_").replace(":", "_")


def extract_image_bytes(row):
    """Extract raw image bytes from a parquet row (handles nested dict or flat column)."""
    if "decoded_image" in row and isinstance(row["decoded_image"], dict):
        return row["decoded_image"].get("bytes")
    if "image" in row and isinstance(row["image"], dict) and "bytes" in row["image"]:
        return row["image"]["bytes"]
    if "image" in row and isinstance(row["image"], bytes):
        return row["image"]
    return row.get("bytes")

# MathVision

## Original

### Setup and Prompt

In [ ]:
# ─── Load MathVision dataset ───
mathvision_df = None
if MATHVISION_PATH.exists():
    mathvision_df = pd.read_parquet(MATHVISION_PATH)
    print(f"✅ Loaded {len(mathvision_df)} rows.")
else:
    print(f"❌ Dataset not found at {MATHVISION_PATH}.")

MATHVISION_OPENROUTER_LIMIT = None  # Set to None for full evaluation

MATHVISION_SYSTEM_PROMPT = """
Answer using only the image and text. 
RULES:
- Output EXACT final answer ONLY.
- NO explanations, reasoning, or filler words.
- If it is a Multiple choice question: output the letter only (A, B, C, D,...).
- Math: output numbers or equations directly.
"""

### Run

In [ ]:
def run_mathvision_openrouter(df, limit=None):
    """Run MathVision evaluation through OpenRouter backend."""
    print(f"\n🚀 MathVision OpenRouter — {MODEL_NAME}...")

    results_dir = Path("MathVisionResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"mathvision_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["id"])

    count = 0
    for _, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break
        task_id = str(row["id"])
        if task_id in processed:
            continue

        level   = str(row.get("level", "N/A"))
        subject = str(row.get("subject", "N/A"))
        query   = str(row["question"])
        gt      = str(row["answer"]).strip()

        print(f"\nTask {task_id} | {subject} | Lvl {level}")

        img_bytes = extract_image_bytes(row)
        b64 = encode_raw_bytes_to_base64(img_bytes) if img_bytes else None
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{MATHVISION_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_math(answer, gt)

        file_exists = append_result({
            "id": task_id, "level": level, "subject": subject,
            "image": str(row.get("image", "N/A")), "question": query,
            "ground_truth": gt, "model_answer": answer,
            "evaluation_passed": passed, "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_id)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
        time.sleep(1)

if mathvision_df is not None:
    run_mathvision_openrouter(mathvision_df, limit=MATHVISION_OPENROUTER_LIMIT)
else:
    print("❌ mathvision_df not loaded — run the Setup cell first.")

## Arabic

### Setup and Prompt

In [ ]:
# ─── Load MathVision dataset ───
mathvision_df = None
if AR_MATHVISION_PATH.exists():
    mathvision_df = pd.read_parquet(AR_MATHVISION_PATH)
    print(f"✅ Loaded {len(mathvision_df)} rows.")
else:
    print(f"❌ Dataset not found at {AR_MATHVISION_PATH}.")

AR_MATHVISION_OPENROUTER_LIMIT = None  # Set to None for full evaluation

AR_MATHVISION_SYSTEM_PROMPT = """
أجب باستخدام الصورة والنص فقط.
قواعد:
- قدم الإجابة النهائية فقط.
- بدون شروحات، تبريرات، أو حشو.
- اذا كان السؤال ذو خيارات متعددة: اكتب الحرف فقط (A, B, C, D).
- الرياضيات: اكتب الأرقام أو المعادلات مباشرة.
"""

### Run

In [ ]:
def run_mathvision_openrouter(df, limit=None):
    """Run Arabic MathVision evaluation through OpenRouter backend."""
    print(f"\n🚀 Arabic MathVision OpenRouter — {MODEL_NAME}...")

    results_dir = Path("MathVisionResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"mathvisionarabic_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["id"])

    count = 0
    for _, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break
        task_id = str(row["id"])
        if task_id in processed:
            continue

        level   = str(row.get("level", "N/A"))
        subject = str(row.get("subject", "N/A"))
        query   = str(row["arabic_question"])
        gt      = str(row["answer"]).strip()

        print(f"\nTask {task_id} | {subject} | Lvl {level}")

        img_bytes = extract_image_bytes(row)
        b64 = encode_raw_bytes_to_base64(img_bytes) if img_bytes else None
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{AR_MATHVISION_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_math(answer, gt)

        file_exists = append_result({
            "id": task_id, "level": level, "subject": subject,
            "image": str(row.get("image", "N/A")), "question": query,
            "ground_truth": gt, "model_answer": answer,
            "evaluation_passed": passed, "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_id)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}'")
        time.sleep(1)

if mathvision_df is not None:
    run_mathvision_openrouter(mathvision_df, limit=AR_MATHVISION_OPENROUTER_LIMIT)
else:
    print("❌ mathvision_df not loaded — run the Setup cell first.")

# ChartQA

In [55]:
def normalize_arabic_text(text):
    """Converts Eastern Arabic numerals to Western, and normalizes Arabic punctuation."""
    arabic_to_western = {
        '٠': '0', '١': '1', '٢': '2', '٣': '3', '٤': '4', 
        '٥': '5', '٦': '6', '٧': '7', '٨': '8', '٩': '9',
        '٪': '%', '،': ',', '٫': '.' 
    }
    trans_table = str.maketrans(arabic_to_western)
    return str(text).translate(trans_table)

def evaluate_chartqa(ans, *gts):
    """Robust ChartQA evaluation handling English, Arabic, visual tolerance, the Year trap, and multiple GT columns."""
    
    ans_norm = normalize_arabic_text(ans)
    ans_clean = str(ans_norm).lower().strip()
    
    # Dynamically gather all valid labels from however many columns are passed
    valid_labels = []
    for gt in gts:
        if pd.isna(gt) or gt is None: # Safely ignore nulls if a column is empty
            continue
        if isinstance(gt, list):
            valid_labels.extend(gt)
        else:
            valid_labels.append(str(gt))
            
    # Iterate through all gathered ground truths
    for label in valid_labels:
        label_norm = normalize_arabic_text(label)
        label_clean = str(label_norm).lower().strip()
        
        # 1. Exact String Match
        if ans_clean == label_clean:
            return True
            
        # 2. Numeric Checks
        ans_num_str = re.sub(r'[\$\£\€\,\%]', '', ans_clean)
        gt_num_str = re.sub(r'[\$\£\€\,\%]', '', label_clean)
        
        try:
            ans_float = float(ans_num_str)
            gt_float = float(gt_num_str)
            
            # --- THE FIX: The Year Trap ---
            # If the ground truth is an integer that looks like a year, force exact match.
            if gt_float.is_integer() and 1800 <= gt_float <= 2100:
                if ans_float == gt_float:
                    return True
                # If it doesn't match exactly, we skip the 5% block below
            else:
                # 5% visual estimation tolerance check for normal data values
                if gt_float == 0:
                    if ans_float == 0: 
                        return True
                elif abs(ans_float - gt_float) / abs(gt_float) <= 0.05:
                    return True
        except ValueError:
            pass 

        # 3. Safe Substring Match 
        if label_clean.isdigit():
            if re.search(fr'\b{label_clean}\b', ans_clean):
                return True
        else:
            if label_clean in ans_clean:
                return True
                
    return False


## Original

In [56]:
# ─── Load ChartQA dataset ───
chartqa_df = None
if CHARTQA_PATH.exists():
    chartqa_df = pd.read_parquet(CHARTQA_PATH)
    print(f"✅ Loaded {len(chartqa_df)} rows.")
    print(f"Columns: {list(chartqa_df.columns)}")
else:
    print(f"❌ Dataset not found at {CHARTQA_PATH}.")

✅ Loaded 32719 rows.
Columns: ['imgname', 'query', 'label', 'type', 'image']


In [57]:
# ─── ChartQA run variables ───
CHARTQA_OPENROUTER_LIMIT = None # Set to None for full evaluation

CHARTQA_SYSTEM_PROMPT = """
Answer using ONLY chart data.
RULES:
- Output ONLY the raw final answer. Zero filler or reasoning.
- Numbers: Use digits + units/currencies if applicable (e.g., 45%, $120). Do not spell out.
- If the question asks a yes/no question: output strictly "Yes" or "No".
- Lists: Comma-separated (e.g., A, B, C).
- Labels: Exact text match from the image.

"""

### OpenRouter

In [58]:
def run_chartqa_openrouter(df, limit=None):
    """Run ChartQA evaluation through OpenRouter backend."""
    print(f"🚀 ChartQA OpenRouter — {MODEL_NAME}...")

    results_dir = Path("ChartQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"chartqa_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["imgname", "query"])

    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break

        imgname  = str(row["imgname"])
        query    = str(row["query"])
        label    = row["label"]
        qa_type  = str(row["type"])
        task_key = f"{imgname}_{query}"

        if task_key in processed:
            continue

        print(f"[{index}] Image: {imgname} | Query: {query[:60]}")

        b64 = encode_image_bytes_to_base64(row["image"])
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{CHARTQA_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_chartqa(answer, label)

        file_exists = append_result({
            "id": str(index), "imgname": imgname, "query": query,
            "ground_truth": str(label), "type": qa_type,
            "model_answer": answer, "evaluation_passed": passed,
            "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_key)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}' | GT: '{label}'")
        time.sleep(1)

# ─── Execute ───
if chartqa_df is not None:
    run_chartqa_openrouter(chartqa_df, limit=CHARTQA_OPENROUTER_LIMIT)
else:
    print("❌ chartqa_df not loaded — run the Setup cell first.")

🚀 ChartQA OpenRouter — google/gemini-3.1-flash-lite-preview...
🔄 Resuming — skipping 32466 already-processed tasks.


## Arabic

In [59]:
# ─── Load ChartQA dataset ───
chartqa_df = None
if AR_CHARTQA_PATH.exists():
    chartqa_df = pd.read_parquet(AR_CHARTQA_PATH)
    chartqa_df['arabic_ground_truth'] = chartqa_df['arabic_ground_truth'].astype(str)
    print(f"✅ Loaded {len(chartqa_df)} rows.")
    print(f"Columns: {list(chartqa_df.columns)}")
else:
    print(f"❌ Dataset not found at {AR_CHARTQA_PATH}.")

✅ Loaded 32719 rows.
Columns: ['imgname', 'query', 'label', 'type', 'image', 'arabic_query', 'arabic_ground_truth']


In [60]:
# ─── ChartQA run variables ───
AR_CHARTQA_OPENROUTER_LIMIT = None # Set to None for full evaluation

AR_CHARTQA_SYSTEM_PROMPT = """
أجب باللغة بالعربية و باستخدام بيانات الرسم البياني فقط.
قواعد:
- اكتب الإجابة النهائية المجردة فقط. بدون حشو أو تفسير.
- الأرقام: استخدم الأرقام + الوحدات/العملات إن وجدت (مثل 45%، $120). لا تكتبها حروفاً.
- "إذا كان السؤال يتطلب إجابة بنعم أو لا فقط: أخرج بدقة "نعم" أو "لا.
- القوائم: افصل بينها بفاصلة (مثل أ، ب، ج).
- التسميات: نسخ دقيق للنص كما يظهر في الصورة.

"""

### OpenRouter

In [61]:
def run_chartqa_openrouter(df, limit=None):
    """Run ChartQA evaluation through OpenRouter backend."""
    print(f"🚀 ChartQA OpenRouter — {MODEL_NAME}...")

    results_dir = Path("ChartQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"chartqaarabic_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    processed, file_exists = load_progress(output_csv, ["imgname", "query"])

    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached.")
            break

        imgname  = str(row["imgname"])
        query    = str(row["arabic_query"])
        label    = row["label"]
        label_ar = row["arabic_ground_truth"]
        qa_type  = str(row["type"])
        task_key = f"{imgname}_{query}"

        if task_key in processed:
            continue

        print(f"[{index}] Image: {imgname} | Query: {query[:60]}")

        b64 = encode_image_bytes_to_base64(row["image"])
        if not b64:
            print("   ⚠️ Skipping — image error.")
            continue

        prompt = f"{query}\n{AR_CHARTQA_SYSTEM_PROMPT}"
        answer, dur = query_openrouter(prompt, b64)
        passed = evaluate_chartqa(answer, label, label_ar)

        file_exists = append_result({
            "id": str(index), "imgname": imgname, "query": query, "ground_truth": str(label),
            "ground_truth_ar": str(label_ar), "type": qa_type,
            "model_answer": answer, "evaluation_passed": passed,
            "run_time": round(dur, 3),
        }, output_csv, file_exists)

        processed.add(task_key)
        count += 1
        print(f"   {'✅' if passed else '❌'} | {dur:.2f}s | '{answer}' | GT: '{label}'")
        time.sleep(1)

# ─── Execute ───
if chartqa_df is not None:
    run_chartqa_openrouter(chartqa_df, limit=AR_CHARTQA_OPENROUTER_LIMIT)
else:
    print("❌ chartqa_df not loaded — run the Setup cell first.")

🚀 ChartQA OpenRouter — google/gemini-3.1-flash-lite-preview...
🔄 Resuming — skipping 32488 already-processed tasks.
[10128] Image: two_col_60328.png | Query: nan
   ❌ | 1.96s | 'يرجى طرح السؤال المتعلق بالرسم البياني.' | GT: '3.8'
[21646] Image: multi_col_101008.png | Query: nan
   ❌ | 1.95s | 'يرجى تحديد السؤال المطلوب للإجابة عليه بناءً على الرسم البياني.' | GT: '1125'
[29907] Image: multi_col_40993.png | Query: nan
   ❌ | 2.90s | '14%, 34%, 14%, 31%, 6%' | GT: '52'
[30313] Image: multi_col_80045.png | Query: nan
   ❌ | 1.68s | 'بناءً على الرسم البياني، يرجى تحديد السؤال الذي تود طرحه.' | GT: '50'


# TurtleBench

### Prompt & Dataset Loader

In [ ]:
TURTLEBENCH_SYSTEM_PROMPT = """
INSTRUCTIONS:
1. Write the Python Turtle graphics code to generate the requested image.
2. Output ONLY the raw Python code with no setup for execution.
3. DO NOT wrap the code in markdown formatting (e.g., do not use ```python).
4. DO NOT output any conversational text, explanations, or greetings.
5. Assume the turtle object is named `t` and the math module is imported if needed.
"""

def load_turtlebench(dataset_path):
    """Parse the official jsonl manifest and load exactly the 260 defined tasks."""
    jsonl_path = Path(dataset_path) / "dataset.jsonl"
    tasks_dir = Path(dataset_path) / "Tasks"
    
    if not jsonl_path.exists():
        print(f"❌ Cannot find {jsonl_path}")
        return pd.DataFrame()

    rows = []
    with open(jsonl_path, 'r', encoding="utf-8-sig") as f:
        for line in f:
            data = json.loads(line)
            
            # Extract directly from the JSON row
            task_id = str(data["id"])
            question_id = f"q{data['question_number']}"
            
            # The input image remains the base shape inside the task folder
            image_path = tasks_dir / task_id / "image" / f"{task_id}.png"
            
            query_text = data.get("query", "")
            variables_text = data.get("variables", "")
            ground_truth_code = data.get("base_shape_code", "No code found.")
            
            # Append critical geometry variables to the prompt
            if variables_text:
                query_text += f"\n\nConstraint - Use these variables strictly:\n{variables_text}"

            rows.append({
                "task_id": task_id,
                "question_id": question_id,
                "image_path": str(image_path),
                "query_text": query_text,
                "ground_truth_code": ground_truth_code
            })
                
    return pd.DataFrame(rows)

### OpenRouter

In [ ]:
# ─── TurtleBench run variables (OpenRouter) ───
TURTLEBENCH_OPENROUTER_LIMIT = None  # Set to None for full evaluation

def run_turtlebench_openrouter(df, limit=None):
    """Run TurtleBench code generation via OpenRouter and save to CSV."""
    print(f"\n🚀 TurtleBench Generation — {MODEL_NAME}...")

    results_dir = Path("TurtleBenchResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    
    safe_name = safe_model_filename(MODEL_NAME)
    output_csv = results_dir / f"turtlebench_{safe_name}_generations.csv"
    
    # Safely load processed keys by combining task and question IDs
    processed_keys = set()
    file_exists = output_csv.exists()
    if file_exists:
        try:
            exist_df = pd.read_csv(output_csv, encoding="utf-8-sig")
            for _, r in exist_df.iterrows():
                processed_keys.add(f"{r['task_id']}_{r['question_id']}")
        except Exception:
            pass

    count = 0  
    skipped_processed_count = 0  # <-- Initialize already-processed counter

    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} questions reached.")
            break

        task_id = str(row["task_id"])
        question_id = str(row["question_id"])
        task_key = f"{task_id}_{question_id}" 

        # Skip if this exact task + question combo is already in the CSV
        if task_key in processed_keys:
            skipped_processed_count += 1
            continue

        # Print how many we skipped right before we start a new one
        if skipped_processed_count > 0:
            print(f"⏭️ Skipped {skipped_processed_count} already-processed tasks.")
            skipped_processed_count = 0 # Reset counter

        print(f"[{index}] Task: {task_id} | Q: {question_id}...")

        query_text = str(row["query_text"])
        image_path = str(row["image_path"])
        ground_truth = str(row["ground_truth_code"])

        b64_url = encode_image_file_to_base64(image_path)
        if not b64_url:
            print("   ⚠️ Skipping — image error.")
            continue

        # PROMPT TUNING
        prompt = f"{TURTLEBENCH_SYSTEM_PROMPT}\n\nTASK:\n{query_text}"
        model_response, gen_time = query_openrouter(prompt, b64_url)

        # Dictionary exactly matches requested columns
        result_dict = {
            "task_id": task_id, 
            "question_id": question_id,
            "image_path": image_path,
            "query_text": query_text,
            "ground_truth_code": ground_truth,
            "model_response": model_response,
            "code_generation_time": round(gen_time, 2)
        }
        
        enc = "utf-8-sig" if not file_exists else "utf-8"
        pd.DataFrame([result_dict]).to_csv(
            output_csv, mode="a", header=not file_exists, index=False,
            encoding=enc,
        )
        file_exists = True
        
        processed_keys.add(task_key)
        count += 1  
        print(f"   ✅ Saved | {gen_time:.2f}s | Task: {task_id} ({question_id})")
        time.sleep(1)

# ─── Execute ───
if TURTLEBENCH_PATH:
    _tb_df = load_turtlebench(TURTLEBENCH_PATH)
    if _tb_df is not None and not _tb_df.empty:
        print(f"✅ Loaded {len(_tb_df)} questions.")
        run_turtlebench_openrouter(_tb_df, limit=TURTLEBENCH_OPENROUTER_LIMIT)
    else:
        print("⚠️ No TurtleBench questions found.")
else:
    print("❌ DTurtleBench folder not found.")

# PEARL

### Prompt & Evaluator

In [ ]:
PEARL_SYSTEM_PROMPT = """
أجب باستخدام الصورة والنص فقط.
أخرج الإجابة الدقيقة فقط. بدون أي حشو أو تفسيرات.
يجب أن تتطابق لغة الإخراج ديناميكيًا مع لغة السؤال تمامًا.
الاختيار من متعدد: أخرج النص الكامل للإجابة، وليس الحرف.
صواب/خطأ: أخرج 'TRUE' أو 'FALSE' فقط.
"""

def evaluate_pearl_anls(ans, gt_text, gt_letter, threshold=0.5):
    """
    Calculates ANLS for PEARL, checking both the full text answer and the answer letter.
    Returns the highest score between the two comparisons.
    """
    ans_str = str(ans).strip().lower()
    gt_text_str = str(gt_text).strip().lower()
    gt_letter_str = str(gt_letter).strip().lower()

    # Helper function to calculate a single ANLS score
    def calculate_score(a_str, g_str):
        if not a_str or not g_str:
            return 1.0 if a_str == g_str else 0.0
        if a_str == g_str:
            return 1.0
            
        dist = Levenshtein.distance(a_str, g_str)
        max_len = max(len(a_str), len(g_str))
        normalized_dist = dist / max_len
        
        if normalized_dist < threshold:
            return round(1.0 - normalized_dist, 3)
        return 0.0

    # 1. Get score against the full text answer
    score_text = calculate_score(ans_str, gt_text_str)
    
    # 2. Get score against the letter (safely ignoring blanks/NaNs from True/False questions)
    score_letter = 0.0
    if gt_letter_str and gt_letter_str != "nan":
        score_letter = calculate_score(ans_str, gt_letter_str)

    # 3. Return whichever score gives the model the most credit
    return max(score_text, score_letter)

### OpenRouter

In [ ]:
# Backfill row_id into existing PEARL result CSVs so future runs resume by unique parquet row.
# The old PEARL runner processed the first parquet row for each annotation_id, so this maps
# existing rows to the first row_id for that annotation_id.
PEARL_RESULTS_DIR = Path("PEARLResults")
PEARL_CREATE_ROW_ID_BACKUPS = True

pearl_row_map_df = pd.read_parquet(PEARL_PATH, columns=["annotation_id"]).reset_index()
pearl_row_map_df = pearl_row_map_df.rename(columns={"index": "row_id"})
pearl_first_row_for_annotation = (
    pearl_row_map_df.drop_duplicates("annotation_id", keep="first")
    .assign(annotation_id=lambda df: df["annotation_id"].astype(str))
    [["annotation_id", "row_id"]]
)

for csv_file in sorted(PEARL_RESULTS_DIR.glob("pearl_openrouter_*_results.csv")):
    df = pd.read_csv(csv_file, encoding="utf-8-sig")
    if "row_id" in df.columns:
        print(f"Skipping {csv_file.name}: row_id already exists.")
        continue

    if "annotation_id" not in df.columns:
        print(f"Skipping {csv_file.name}: no annotation_id column.")
        continue

    if PEARL_CREATE_ROW_ID_BACKUPS:
        backup_file = csv_file.with_suffix(".before_row_id_backfill.csv")
        if not backup_file.exists():
            csv_file.replace(backup_file)
            df = pd.read_csv(backup_file, encoding="utf-8-sig")
            print(f"Backup created: {backup_file.name}")

    df["annotation_id"] = df["annotation_id"].astype(str)
    df = df.merge(pearl_first_row_for_annotation, on="annotation_id", how="left")

    missing_row_ids = df["row_id"].isna().sum()
    if missing_row_ids:
        print(f"WARNING: {csv_file.name} has {missing_row_ids} rows that could not be mapped to row_id.")

    ordered_columns = ["row_id"] + [col for col in df.columns if col != "row_id"]
    df[ordered_columns].to_csv(csv_file, index=False, encoding="utf-8-sig")
    print(f"Added row_id to {csv_file.name}; rows = {len(df)}")


In [ ]:
PEARL_OPENROUTER_LIMIT = None   # Set to None for full evaluation
PEARL_BATCH_SIZE = 50   # Memory-safe batch size


def run_pearl_openrouter(parquet_path, limit=None):
    """Run PEARL evaluation through OpenRouter using row_id-based resume."""
    if not parquet_path.exists():
        print(f"❌ Dataset not found at {parquet_path}")
        return

    print(f"\n🚀 PEARL OpenRouter — {MODEL_NAME}...")

    results_dir = Path("PEARLResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"pearl_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"

    # Resume by unique parquet row_id, not annotation_id. annotation_id is not unique in PEARL.
    processed, file_exists = load_progress(output_csv, ["row_id"])

    if not file_exists:
        headers = [
            "row_id", "question", "choices", "answer", "answer_letter", "category",
            "country", "image_id", "question_type", "annotation_id",
            "model_answer", "Evaluation", "run_time"
        ]
        pd.DataFrame(columns=headers).to_csv(output_csv, index=False, encoding="utf-8-sig")
        file_exists = True

    count = 0
    limit_reached = False
    row_offset = 0
    parquet_file = pq.ParquetFile(parquet_path)

    for batch in parquet_file.iter_batches(batch_size=PEARL_BATCH_SIZE):
        if limit_reached:
            break

        chunk_df = batch.to_pandas()

        if "original_image" in chunk_df.columns:
            chunk_df = chunk_df.rename(columns={"original_image": "image"})

        for local_idx, row in chunk_df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached.")
                limit_reached = True
                break

            row_id = str(row_offset + local_idx)
            task_id = str(row.get("annotation_id", "N/A"))

            if row_id in processed:
                continue

            query         = str(row.get("question", "")).strip()
            gt            = str(row.get("answer", "")).strip()
            ans_letter    = str(row.get("answer_letter", "")).strip()
            category      = str(row.get("category", "N/A"))
            country       = str(row.get("country", "N/A"))
            image_id      = str(row.get("image_id", "N/A"))
            question_type = str(row.get("question_type", "N/A"))

            choices_raw = row.get("choices", "")
            choices_str = ""

            if isinstance(choices_raw, (list, np.ndarray)):
                choices_str = "\n".join([str(c) for c in choices_raw])
            else:
                choices_str = str(choices_raw).strip()
                if choices_str.startswith("[") and choices_str.endswith("]"):
                    try:
                        parsed_list = ast.literal_eval(choices_str)
                        choices_str = "\n".join([str(c) for c in parsed_list])
                    except (ValueError, SyntaxError):
                        pass

            print(f"\nRow {row_id} | Task {task_id} | image_id: {image_id} | type: {question_type}")

            img_bytes = extract_image_bytes(row)
            b64 = encode_image_bytes_to_base64(img_bytes) if img_bytes else None
            if not b64:
                print("   ⚠️ Skipping — image error.")
                continue

            if question_type == "multiple_choice" and choices_str and choices_str not in ("[]", "nan"):
                prompt = f"{query}\n\nOptions:\n{choices_str}\n\n{PEARL_SYSTEM_PROMPT}"
            else:
                prompt = f"{query}\n\n{PEARL_SYSTEM_PROMPT}"

            answer, dur = query_openrouter(prompt, b64)
            score = evaluate_pearl_anls(answer, gt, ans_letter)

            result_dict = {
                "row_id": row_id,
                "question": query,
                "choices": str(choices_raw),
                "answer": gt,
                "answer_letter": ans_letter,
                "category": category,
                "country": country,
                "image_id": image_id,
                "question_type": question_type,
                "annotation_id": task_id,
                "model_answer": answer,
                "Evaluation": score,
                "run_time": round(dur, 2),
            }

            file_exists = append_result(result_dict, output_csv, file_exists)
            processed.add(row_id)
            count += 1

            print(f"   Score: {score:.3f} | {dur:.2f}s | Ans: '{answer}' (GT: '{gt}' / '{ans_letter}')")
            time.sleep(1)

        row_offset += len(chunk_df)


if PEARL_PATH:
    run_pearl_openrouter(PEARL_PATH, limit=PEARL_OPENROUTER_LIMIT)
else:
    print("❌ PEARL_PATH not defined — check your paths.")


# JEEM

### Prompts

In [ ]:
JEEM_GENERATION_PROMPT = """
أجب عن الأسئلة المرفقة بناءً على الصورة فقط، باستخدام اللهجة: {dialect}.
أخرج إجاباتك بصيغة JSON حصراً باستخدام هذه المفاتيح: {expected_keys}.
يجب أن يحتوي الـ JSON على الإجابات فقط بدون أي نص إضافي، أو مقدمات، أو شروحات.

مثال للتنسيق المطلوب:
{{
  "q1": "إجابة السؤال الأول هنا",
  "q3": "إجابة السؤال الثالث هنا"
}}

"""

JEEM_JUDGE_PROMPT = """
أنت مقيم لغوي صارم. مهمتك هي مقارنة إجابات النموذج بالإجابات الصحيحة للأسئلة المرفقة.
اللهجة المستخدمة: {dialect}.

القواعد الجوهرية:
1. قيم كل إجابة على حدة. الإجابة التي تحمل نفس المعنى الدقيق للإجابة الصحيحة تعتبر صحيحة.
2. احسب إجمالي عدد الإجابات الصحيحة من إجمالي الأسئلة المرفقة.
3. يجب أن يكون المخرج حصرياً بصيغة JSON متوافقة، بدون أي نص أو شروحات إضافية.

مثال للمخرج المطلوب:
{{
  "is_perfect": false,
  "score": "4 outof 5",
}}

{{
  "is_perfect": true,
  "score": "5 outof 5",
}}
"""

### OpenRouter

In [ ]:
JEEM_BATCH_SIZE = 20
JEEM_LIMIT = None  # Set to an integer (e.g., 5) to test a small run first

def run_jeem_generation_batched(parquet_path, limit=None):
    if not parquet_path.exists():
        print(f"❌ Dataset not found at {parquet_path}")
        return

    print(f"\n🚀 JEEM Batched Generation — {MODEL_NAME}...")

    results_dir = Path("JEEMResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"jeem_gen_{safe_model_filename(MODEL_NAME)}.csv"
    
    # Track progress using 'id'
    processed, file_exists = load_progress(output_csv, ["id"])

    # Pre-initialize the file to protect Arabic encoding
    if not file_exists:
        # --- UPDATED: Added combined_ground_truth to headers ---
        headers = ["id", "dialect", "combined_questions", "combined_ground_truth", "combined_model_answers", "run_time"]
        pd.DataFrame(columns=headers).to_csv(output_csv, index=False, encoding="utf-8-sig")
        file_exists = True

    count = 0
    limit_reached = False
    parquet_file = pq.ParquetFile(parquet_path)
    
    # Memory-safe batch streaming
    for batch in parquet_file.iter_batches(batch_size=JEEM_BATCH_SIZE):
        if limit_reached: 
            break
            
        chunk_df = batch.to_pandas()
        
        # Map the image column safely for the helper function
        if "original_image" in chunk_df.columns:
            chunk_df = chunk_df.rename(columns={"original_image": "image"})
        
        for _, row in chunk_df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached.")
                limit_reached = True
                break

            task_id = str(row.get("id", "N/A"))
            if task_id in processed:
                continue

            dialect = str(row.get("dialect", "Modern Standard Arabic")).strip()

            # --- 1. Filter Questions & Extract Ground Truth based on 'no_answer' flag ---
            valid_qs = {}
            valid_gts = {}  # --- NEW: Dictionary to hold valid ground truths ---
            for i in range(1, 6):
                no_ans_val = str(row.get(f"no_answer{i}", "")).strip().lower()
                # Skip if the column value indicates it's unanswerable
                is_no_answer = no_ans_val in ['true', '1', 'yes']
                
                if not is_no_answer:
                    q_text = str(row.get(f"question{i}", "")).strip()
                    gt_text = str(row.get(f"answer{i}", "")).strip()  # --- NEW: Grab the correct answer ---
                    
                    if q_text and q_text != "nan":
                        valid_qs[f"q{i}"] = q_text
                        valid_gts[f"q{i}"] = gt_text

            # If all 5 questions were marked 'no_answer', skip calling the model
            if not valid_qs:
                print(f"⏭️ Task {task_id} skipped (all questions marked as no_answer).")
                processed.add(task_id)
                continue

            # --- 2. Format the Token-Optimized Prompt ---
            expected_keys_str = ", ".join([f'"{ k}"' for k in valid_qs.keys()])
            qs_text = "\n".join([f"سؤال {k.replace('q','')}: {v}" for k, v in valid_qs.items()])
            
            sys_prompt = JEEM_GENERATION_PROMPT.format(
                dialect=dialect, 
                expected_keys=expected_keys_str
            )
            final_prompt = f"{sys_prompt}\n\nالأسئلة:\n{qs_text}"

            # --- 3. Extract and Process Image ---
            img_bytes = extract_image_bytes(row)
            b64 = encode_image_bytes_to_base64(img_bytes) if img_bytes else None
            if not b64:
                print(f"⚠️ Skipping Task {task_id} — image error.")
                continue

            # --- 4. Query the Model ---
            print(f"\nTask {task_id} | Dialect: {dialect} | Valid Questions: {len(valid_qs)}/5")
            model_response, dur = query_openrouter(final_prompt, b64)
            
            # --- 5. Parse JSON & Combine Answers ---
            model_answers_dict = {}
            try:
                # Strip markdown blocks safely
                clean_json = model_response.strip("` \n").replace("json\n", "")
                model_answers_dict = json.loads(clean_json)
            except Exception as e:
                print(f"   ⚠️ JSON Parse Error: {e}")
                # Fallback to save raw text to avoid losing the generation
                model_answers_dict = {k: f"[PARSE ERROR] Raw: {model_response}" for k in valid_qs.keys()}

            # Combine valid questions, ground truth, and model answers with Q/A prefixes
            combined_questions = "\n".join([f"Q{k.replace('q','')}: {v}" for k, v in valid_qs.items()])
            
            # --- UPDATED: Combine the ground truth answers ---
            combined_ground_truth = "\n".join([f"A{k.replace('q','')}: {v}" for k, v in valid_gts.items()])
            
            combined_answers = "\n".join([f"A{k.replace('q','')}: {model_answers_dict.get(k, 'MISSING')}" for k in valid_qs.keys()])

            # --- 6. Save the Results ---
            result_dict = {
                "id": task_id,
                "dialect": dialect,
                "combined_questions": combined_questions,
                "combined_ground_truth": combined_ground_truth,  # --- NEW: Added to final dict ---
                "combined_model_answers": combined_answers,
                "run_time": round(dur, 2)
            }
            
            file_exists = append_result(result_dict, output_csv, file_exists)
            processed.add(task_id)
            count += 1
            
            print(f"   ✅ Saved {len(valid_qs)} answers | {dur:.2f}s")
            time.sleep(1)

# --- Execution ---
if JEEM_PATH:
    run_jeem_generation_batched(JEEM_PATH, limit=JEEM_LIMIT)
else:
    print("❌ JEEM_PATH not defined.")

# ScreenQA

### Prompt & Evaluator

In [ ]:
# SCREENQA_SYSTEM_PROMPT = """
# Answer using ONLY the screenshot.
# 1. EXACT MATCH: Copy UI text exactly (case/punctuation/spelling). No synonyms.
# 2. NUMBERS: Output digits (e.g., "4") unless explicitly spelled out on screen.
# 3. Output ONLY "Yes" or "No" for boolean queries.
# 4. Output ONLY the exact answer. Zero explanations or filler.
# """

# # ─── Image resize dimensions for ScreenQA ───
# SCREENQA_RESIZE = (1280, 720)


# def evaluate_screenqa(ans, gt, threshold=0.5):
#     """Calculates ANLS (Average Normalized Levenshtein Similarity) for ScreenQA."""
#     # 1. Clean strings (lowercase, strip whitespace, remove currencies)
#     ans_str = re.sub(r'[\$\£\€\,]', '', str(ans).lower()).strip()
#     gt_str  = re.sub(r'[\$\£\€\,]', '', str(gt).lower()).strip()

#     # 2. Handle empty strings safely
#     if not ans_str or not gt_str:
#         return 1.0 if ans_str == gt_str else 0.0

#     # 3. Calculate Levenshtein Distance
#     dist = Levenshtein.distance(ans_str, gt_str)
    
#     # 4. Normalize by the maximum length
#     max_len = max(len(ans_str), len(gt_str))
#     normalized_dist = dist / max_len

#     # 5. Apply the threshold (default 0.5)
#     # If the error rate is under 50%, give partial credit. Otherwise, 0 score.
#     if normalized_dist < threshold:
#         return round(1.0 - normalized_dist, 2)
#     else:
#         return 0.0

# def extract_screenqa_ground_truth(row):
#     """
#     Robust extractor using Regex to find the short 'text' answer within 
#     nested, stringified, or NumPy-wrapped structures.
#     """
#     # 1. Pull the raw data and force it into a string format for searching
#     raw_str = str(row.get("answers", row.get("ground_truth", row.get("full_answer", ""))))

#     # 2. PRIORITY: Extract the specific UI element text (e.g., "$70", "Grace", "Free")
#     # This looks for 'text': followed by a quote, and captures everything inside the quotes.
#     text_match = re.search(r"'text':\s*['\"]([^'\"]+)['\"]", raw_str)
#     if text_match:
#         return text_match.group(1).strip()

#     # 3. FALLBACK: Extract the 'full_answer' conversational string if 'text' isn't found
#     full_match = re.search(r"'full_answer':\s*['\"]([^'\"]+)['\"]", raw_str)
#     if full_match:
#         return full_match.group(1).strip()

#     # 4. FINAL FALLBACK: Return the cleaned raw string
#     return raw_str.strip()

### OpenRouter

In [ ]:
# SCREENQA_OPENROUTER_LIMIT = 2  # Set to None for full evaluation
# SCREENQA_BATCH_SIZE = 50  # Number of rows to load into RAM at a time

# def run_screenqa_openrouter(parquet_path, limit=None):
#     """Run ScreenQA evaluation through OpenRouter using memory-safe chunking."""
#     if not parquet_path.exists():
#         print(f"❌ Dataset not found at {parquet_path}.")
#         return

#     print(f"\n🚀 ScreenQA OpenRouter — {MODEL_NAME}...")

#     results_dir = Path("ScreenQAResults")
#     results_dir.mkdir(parents=True, exist_ok=True)
#     output_csv = results_dir / f"screenqa_openrouter_{safe_model_filename(MODEL_NAME)}_results.csv"
    
#     # Load progress to resume safely
#     processed, file_exists = load_progress(output_csv, ["id"])

#     # Open the Parquet file as a stream rather than loading it all
#     parquet_file = pq.ParquetFile(parquet_path)
#     print(f"📊 Total dataset rows: {parquet_file.metadata.num_rows}")
    
#     count = 0
#     limit_reached = False
    
#     # Loop through the dataset in safe batches
#     for batch_idx, batch in enumerate(parquet_file.iter_batches(batch_size=SCREENQA_BATCH_SIZE)):
#         if limit_reached:
#             break
            
#         print(f"\n📦 Loading Batch {batch_idx + 1} into RAM...")
#         chunk_df = batch.to_pandas()
        
#         for _, row in chunk_df.iterrows():
#             # Check limit
#             if limit and count >= limit:
#                 print(f"🎯 Limit of {limit} reached.")
#                 limit_reached = True
#                 break

#             task_id = str(row["screen_id"])
            
#             # Skip already processed
#             if task_id in processed:
#                 continue

#             image_name = str(row.get("file_name", "N/A"))
#             query = str(row["question"])
#             gt = extract_screenqa_ground_truth(row)

#             print(f"\nTask {task_id} | Image: {image_name}")

#             img_bytes = extract_image_bytes(row)
#             b64 = encode_image_bytes_to_base64(img_bytes, resize=SCREENQA_RESIZE) if img_bytes else None
#             if not b64:
#                 print("   ⚠️ Skipping — image error.")
#                 continue

#             prompt = f"{query}\n{SCREENQA_SYSTEM_PROMPT}"
#             answer, dur = query_openrouter(prompt, b64)
            
#             # Calculate ANLS Score (using the integrated function from earlier)
#             passed = evaluate_screenqa(answer, gt)

#             file_exists = append_result({
#                 "id": task_id, "image": image_name, "question": query,
#                 "ground_truth": gt, "model_answer": answer,
#                 "evaluation_passed": passed, "run_time": round(dur, 2),
#             }, output_csv, file_exists)

#             processed.add(task_id)
#             count += 1
            
#             # If using ANLS, 'passed' will be a float score. Format it nicely.
#             score_display = f"{passed:.3f}" if isinstance(passed, float) else ('✅' if passed else '❌')
#             print(f"   Score: {score_display} | {dur:.2f}s | Ans: '{answer}' (GT: '{gt}')")
            
#             time.sleep(1)
            
#         # Once this loop finishes, `chunk_df` is garbage collected. RAM is instantly freed!

# # Notice we are passing the PATH directly now, not the pre-loaded dataframe
# if SCREENQA_PATH:
#     run_screenqa_openrouter(SCREENQA_PATH, limit=SCREENQA_OPENROUTER_LIMIT)
# else:
#     print("❌ SCREENQA_PATH is not defined.")

# Results Evaluation

In [ ]:
# JUDGE_MODEL = "openai/gpt-4o-mini"   Not decided  
JUDGE_MODEL = None  

### ChartQA Eval

In [ ]:
from datetime import datetime

CHARTQA_RESULTS_DIR = Path("ChartQAResults")
CHARTQA_NUMERIC_TOLERANCE = 0.05
CHARTQA_CREATE_BACKUP = False


def chartqa_eval_normalize_text(value):
    """Normalize text for ChartQA answer comparison, including Arabic digits/punctuation."""
    if pd.isna(value):
        return ""

    text = normalize_arabic_text(str(value)).lower().strip()
    text = text.replace("\u00a0", " ").replace("\u202f", " ")
    text = re.sub(r"[`*_~\"'،؛:;!?()\[\]{}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def chartqa_eval_compact_number_text(value):
    """Remove separators inside digits so values like '40 040' and '40,040' become '40040'."""
    text = normalize_arabic_text(str(value)).lower()
    text = text.replace("\u00a0", " ").replace("\u202f", " ")
    previous = None
    while previous != text:
        previous = text
        text = re.sub(r"(?<=\d)[\s,_](?=\d)", "", text)
    return text


def chartqa_eval_extract_labels(*values):
    """Collect valid ground-truth labels from scalar, list-like, or bracketed string cells."""
    labels = []
    for value in values:
        if value is None or pd.isna(value):
            continue
        if isinstance(value, (list, tuple, set, np.ndarray)):
            labels.extend(value)
            continue

        text = str(value).strip()
        if not text or text.lower() == "nan":
            continue

        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, (list, tuple, set)):
                    labels.extend(parsed)
                    continue
            except Exception:
                labels.extend(part.strip() for part in text.strip("[]").split(",") if part.strip())
                continue

        labels.append(text)
    return [str(label).strip() for label in labels if str(label).strip()]


def chartqa_eval_numbers(value):
    """Extract comparable numeric values after fixing grouped digits and simple percent/currency formatting."""
    text = chartqa_eval_compact_number_text(value)
    text = re.sub(r"[^0-9.\-+% ]", " ", text)
    numbers = []
    for match in re.finditer(r"[-+]?\d+(?:\.\d+)?%?", text):
        raw = match.group(0).rstrip("%")
        try:
            numbers.append(float(raw))
        except ValueError:
            pass
    return numbers


def chartqa_eval_numeric_match(model_answer, ground_truth, tolerance=CHARTQA_NUMERIC_TOLERANCE):
    ans_nums = chartqa_eval_numbers(model_answer)
    gt_nums = chartqa_eval_numbers(ground_truth)
    if not ans_nums or not gt_nums:
        return False

    for gt_num in gt_nums:
        for ans_num in ans_nums:
            if gt_num == ans_num:
                return True

            # Years should match exactly; a 5% tolerance is too loose for dates.
            if float(gt_num).is_integer() and 1800 <= gt_num <= 2100:
                continue

            if gt_num == 0:
                if ans_num == 0:
                    return True
            elif abs(ans_num - gt_num) / abs(gt_num) <= tolerance:
                return True
    return False


def chartqa_eval_string_match(model_answer, ground_truth):
    ans = chartqa_eval_normalize_text(model_answer)
    gt = chartqa_eval_normalize_text(ground_truth)
    if not ans or not gt:
        return False

    if ans == gt:
        return True

    ans_compact = chartqa_eval_compact_number_text(ans)
    gt_compact = chartqa_eval_compact_number_text(gt)
    if ans_compact == gt_compact:
        return True

    # Allow full-label containment for textual answers, but avoid very short accidental matches.
    if not re.fullmatch(r"[-+]?\d+(?:\.\d+)?", gt_compact):
        if len(gt) >= 3 and re.search(rf"(?<!\w){re.escape(gt)}(?!\w)", ans):
            return True
    return False


def evaluate_chartqa_robust(model_answer, *ground_truth_values):
    """Re-evaluate ChartQA rows with stronger numeric grouping, Arabic, list, and text handling."""
    labels = chartqa_eval_extract_labels(*ground_truth_values)
    for label in labels:
        if chartqa_eval_string_match(model_answer, label):
            return True
        if chartqa_eval_numeric_match(model_answer, label):
            return True
    return False


def chartqa_is_false(value):
    return str(value).strip().lower() in {"false", "0", "0.0", "no", "nan", ""}


def revaluate_chartqa_false_rows(results_dir=CHARTQA_RESULTS_DIR):
    """Re-read every ChartQA CSV and update only rows currently marked False in evaluation_passed."""
    results_dir = Path(results_dir)
    csv_files = sorted(results_dir.glob("*.csv"))
    print(f"Found {len(csv_files)} ChartQA result CSV files in {results_dir}.")

    summary = []
    backup_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    for csv_file in csv_files:
        df = pd.read_csv(csv_file, encoding="utf-8-sig")
        required = {"model_answer", "evaluation_passed"}
        if not required.issubset(df.columns) or "ground_truth" not in df.columns:
            print(f"Skipping {csv_file.name}: missing required columns.")
            continue

        false_mask = df["evaluation_passed"].apply(chartqa_is_false)
        false_before = int(false_mask.sum())
        fixed_count = 0

        for idx in df.index[false_mask]:
            row = df.loc[idx]
            gt_values = [row.get("ground_truth")]
            if "ground_truth_ar" in df.columns:
                gt_values.append(row.get("ground_truth_ar"))

            if evaluate_chartqa_robust(row.get("model_answer"), *gt_values):
                df.at[idx, "evaluation_passed"] = True
                fixed_count += 1

        if fixed_count:
            if CHARTQA_CREATE_BACKUP:
                backup_file = csv_file.with_suffix(f".before_chartqa_reeval_{backup_stamp}.csv")
                csv_file.replace(backup_file)
                df.to_csv(csv_file, index=False, encoding="utf-8-sig")
            else:
                df.to_csv(csv_file, index=False, encoding="utf-8-sig")

        false_after = false_before - fixed_count
        summary.append({
            "file": csv_file.name,
            "rows": len(df),
            "false_before": false_before,
            "fixed_false_rows": fixed_count,
            "false_after": false_after,
        })
        print(f"{csv_file.name}: fixed {fixed_count} of {false_before} false rows.")

    return pd.DataFrame(summary)


chartqa_reeval_summary = revaluate_chartqa_false_rows()
chartqa_reeval_summary


### MathVision Eval

In [ ]:
TARGET_FOLDER_PATH = "MathVisionResults" # Folder containing all your raw CSVs
OUTPUT_CSV_SUFFIX = "_GRADED.csv"        # What to append to the saved file
JUDGETEMP = 0.0      # Keep at 0.0 for deterministic True/False grading

def llm_math_judge_openrouter(question, model_answer, ground_truth):
    """Uses OpenRouter to verify mathematical equivalence with question context."""
    prompt = f"""You are an expert math grader. Analyze if the Model Answer is mathematically equivalent to the Ground Truth based on the Question context.

Question: {question}
Model Answer: {model_answer}
Ground Truth: {ground_truth}

Answer ONLY with "True" or "False". Do not explain your reasoning.
"""
    
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": JUDGE_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": JUDGETEMP,
    }
    
    try:
        response = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers=headers, json=payload, timeout=120,
        )
        response.raise_for_status()
        result_text = response.json()["choices"][0]["message"]["content"].strip().lower()
        return "true" in result_text
    except Exception as e:
        print(f"   ⚠️ OpenRouter Judge Error: {e}")
        return False

def evaluate_mathvision_row(question, ans, gt):
    """Layered evaluation: Fast programmatic checks first, OpenRouter fallback second."""
    ans_str = str(ans).strip().lower()
    gt_str = str(gt).strip().lower()

    # --- LAYER 1: FAST PROGRAMMATIC CHECKS ---
    if ans_str == gt_str:
        return True

    if gt_str in ['a', 'b', 'c', 'd', 'e']:
        if re.search(fr'\b{gt_str}\b', ans_str):
            return True
        return False 

    try:
        if float(ans_str) == float(gt_str):
            return True
    except ValueError:
        pass 
        
    ans_clean = ans_str.replace(" ", "").replace("$", "")
    gt_clean = gt_str.replace(" ", "").replace("$", "")
    
    if ans_clean == gt_clean:
        return True

    # --- LAYER 2: OPENROUTER JUDGE ---
    # Triggered if complex math symbols exist and programmatic checks fail
    if any(char in gt_str for char in ['\\', '/', '+', '-', '*', '^', '=', 'x', 'y', 'pi', 'sqrt']):
        return llm_math_judge_openrouter(question, ans_str, gt_str)

    return False

def grade_results_csv(csv_path):
    print(f"\n🚀 Starting Post-Processing Evaluation on {csv_path.name} using {JUDGE_MODEL}...")
    
    try:
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
    except Exception as e:
        print(f"❌ Error reading file {csv_path.name}: {e}")
        return
        
    # Ensure columns exist
    if not all(col in df.columns for col in ['question', 'model_answer', 'ground_truth']):
        print(f"❌ CSV {csv_path.name} is missing required columns ('question', 'model_answer', 'ground_truth'). Skipping.")
        return

    judgments = []
    for row in df.itertuples():
        question = str(row.question)
        ans = str(row.model_answer)
        gt = str(row.ground_truth)
        
        is_correct = evaluate_mathvision_row(question, ans, gt)
        judgments.append(is_correct)
        
        mark = "✅" if is_correct else "❌"
        # Print a short log so you can watch it work
        print(f"Row {row.Index} | {mark} | Model: {ans[:20]}... | GT: {gt[:20]}...")

    # Attach the final verdicts back to the DataFrame under the new column name
    df['ComplexEval'] = judgments
    
    # Save a clean, graded version of the CSV
    graded_csv_path = csv_path.parent / f"{csv_path.stem}{OUTPUT_CSV_SUFFIX}"
    df.to_csv(graded_csv_path, index=False, encoding="utf-8-sig")
    
    # Calculate final accuracy for your thesis
    accuracy = (sum(judgments) / len(judgments)) * 100
    print(f"\n📊 Final Accuracy for {csv_path.name}: {accuracy:.2f}%")
    print(f"💾 Graded results saved to: {graded_csv_path.name}")

def process_folder():
    """Iterates through all CSV files in the target folder and processes them."""

    folder = Path(TARGET_FOLDER_PATH)
    if not folder.exists() or not folder.is_dir():
        print(f"❌ Folder not found: {TARGET_FOLDER_PATH}")
        return

    # Find all CSV files, excluding ones that already have the graded suffix
    csv_files = [f for f in folder.glob("*.csv") if not f.name.endswith(OUTPUT_CSV_SUFFIX)]

    if not csv_files:
        print(f"⚠️ No ungraded CSV files found in '{TARGET_FOLDER_PATH}'.")
        return

    print(f"📂 Found {len(csv_files)} CSV file(s) to process in '{TARGET_FOLDER_PATH}'.")
    
    for csv_file in csv_files:
        grade_results_csv(csv_file)
        
    print("\n🎉 All files processed successfully!")

if __name__ == "__main__":
    process_folder()


### TurtleBench Eval

In [ ]:
RESULTS_FOLDER = "TurtleBenchResults"

TURTLEBENCH_EVAL_PROMPT_TEMPLATE = """
Compare the following two Python Turtle scripts.
Analyze their drawing logic and determine if they will generate the exact same visual shape/output on the screen.
CRITICAL RULE: You must ignore missing setup code, imports (like 'import turtle'), or initialization (like 't = turtle.Turtle()'). 
Focus ONLY on the geometric drawing logic. If the core logic produces the same final shape, you MUST answer 'Yes'.
Answer strictly with 'Yes' or 'No' followed by a single dash '-' and one brief statement explaining your reasoning. Do not exceed one sentence.

Model Generated Code:
{model_code}

Ground Truth Code:
{ground_truth_code}

"""

def evaluate_turtle_equivalence_openrouter(model_code, gt_code):
    """Use OpenRouter to judge Turtle script equivalence."""
    prompt = TURTLEBENCH_EVAL_PROMPT_TEMPLATE.format(
        model_code=model_code, ground_truth_code=gt_code
    )
    
    for attempt in range(MAX_RETRIES):
        try:
            response_text, _ = query_openrouter_text(prompt, model_name=JUDGE_MODEL)
            
            raw_lower = response_text.lower()
            
            # Helper to extract just the reasoning portion after the dash
            def extract_reason(text):
                if '-' in text:
                    return text.split('-', 1)[1].strip()
                return text

            if raw_lower.startswith("yes"):
                return True, extract_reason(response_text)
            elif raw_lower.startswith("no"):
                return False, extract_reason(response_text)
            
            if "yes" in raw_lower:
                return True, extract_reason(response_text)
            if "no" in raw_lower:
                return False, extract_reason(response_text)
                
            return "ERROR", response_text
            
        except Exception as e:
            print(f"   ❌ Eval attempt {attempt + 1}/{MAX_RETRIES}: {e}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(5 * (attempt + 1))
                
    return "ERROR", "Max retries reached."

def evaluate_all_results(results_dir):
    """Iterates through all unjudged CSVs in the results folder and evaluates them."""
    target_dir = Path(results_dir)
    if not target_dir.exists():
        print(f"❌ Directory not found: {target_dir}")
        return

    # Find all CSVs but ignore ones that have already been judged
    all_csvs = list(target_dir.glob("*.csv"))
    csvs_to_process = [f for f in all_csvs if not f.stem.endswith("_LLM_JUDGED")]

    if not csvs_to_process:
        print("✅ No new CSVs to evaluate. All caught up!")
        return

    print(f"🚀 Starting LLM-as-a-Judge Evaluation using {JUDGE_MODEL} on {len(csvs_to_process)} file(s)...")

    for csv_path in csvs_to_process:
        print(f"\n📄 Processing: {csv_path.name}")
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
        
        judgments = []
        reasonings = []
        
        for row in df.itertuples():
            task_id = getattr(row, 'task_id', f"Row_{row.Index}")
            print(f"Evaluating Task {task_id}...")
            
            model_code = str(getattr(row, 'model_response', ''))
            gt_code = str(getattr(row, 'ground_truth_code', ''))
            
            # Handle empty/missing code gracefully (checking for 'nan' from pandas)
            if not model_code.strip() or not gt_code.strip() or model_code == 'nan' or gt_code == 'nan':
                print("   ⚠️ Missing model or ground truth code. Skipping.")
                judgments.append(False)
                reasonings.append("Missing code for comparison.")
                continue
            
            is_equivalent, explanation = evaluate_turtle_equivalence_openrouter(
                model_code=model_code, 
                gt_code=gt_code
            )
            
            judgments.append(is_equivalent)
            reasonings.append(explanation)
            
            print(f"   {'✅' if is_equivalent is True else '❌' if is_equivalent is False else '⚠️'} Judge Result: {is_equivalent}")
            time.sleep(1)
            
        # These two lines ensure the CSV gets exactly the two requested columns
        df['LLM_Judge_Result'] = judgments
        df['LLM_Judge_Reasoning'] = reasonings
        
        output_name = csv_path.parent / f"{csv_path.stem}_LLM_JUDGED.csv"
        df.to_csv(output_name, index=False, encoding="utf-8-sig")
        print(f"💾 File complete! Saved to: {output_name.name}")

evaluate_all_results(
    results_dir=RESULTS_FOLDER
)

### JEEM Eval

In [ ]:
TARGET_FOLDER = Path("JEEMResults")      
JUDGE_MAX_RETRIES = 3

def runfolder_evaluation():
    if not TARGET_FOLDER.exists():
        print(f"❌ Target folder {TARGET_FOLDER} does not exist.")
        return
    
    csv_files = [f for f in TARGET_FOLDER.glob("*.csv") if not f.name.startswith("eval_")]
    
    if not csv_files:
        print(f"⚠️ No raw generation CSVs found to evaluate in {TARGET_FOLDER}.")
        return

    print(f"📂 Found {len(csv_files)} original CSV files to evaluate using {JUDGE_MODEL}.")

    for csv_path in csv_files:
        print(f"\n⚖️ Evaluating file: {csv_path.name}")
        
        output_csv = TARGET_FOLDER / f"eval_{csv_path.name}"
        processed, file_exists = load_progress(output_csv, ["id"])
        
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
        
        count = 0
        for _, row in df.iterrows():
            task_id = str(row["id"])
            if task_id in processed:
                continue

            dialect = str(row.get("dialect", "Modern Standard Arabic"))
            questions = str(row.get("combined_questions", ""))
            model_ans = str(row.get("combined_model_answers", ""))
            ground_truth = str(row.get("combined_ground_truth", "")) 

            # Format the payload for the Judge
            judge_sys_prompt = JEEM_JUDGE_PROMPT.format(dialect=dialect)
            evaluation_payload = f"الأسئلة:\n{questions}\n\nالإجابات الصحيحة:\n{ground_truth}\n\nإجابات النموذج:\n{model_ans}"
            final_prompt = f"{judge_sys_prompt}\n\n{evaluation_payload}"

            print(f"   Task {task_id} | Judging...")
            response, dur = query_openrouter_text(final_prompt, model_name=JUDGE_MODEL, max_retries=JUDGE_MAX_RETRIES)
            
            # --- NEW: Safely parse the Judge JSON ---
            is_perfect = False
            score_fraction = "0/0"
            
            try:
                # Strip markdown blocks if the model adds them
                clean_json = response.strip("` \n").replace("json\n", "")
                judge_data = json.loads(clean_json)
                
                is_perfect = judge_data.get("is_perfect", False)
                score_fraction = judge_data.get("score", "Error")
                
                print(f"   ✅ Score: {score_fraction} | Perfect: {is_perfect} | {dur:.2f}s")
                
            except Exception as e:
                print(f"   ⚠️ Judge JSON Parse Error: {e}\nRaw Response: {response}")
                score_fraction = "Parse Error"

            # 3. Add the new metrics to the row dictionary
            result_dict = row.to_dict()
            result_dict["is_perfect"] = is_perfect
            result_dict["score"] = score_fraction
            result_dict["judge_run_time"] = round(dur, 2)
            
            # Append locally using utf-8-sig to protect Arabic encoding
            enc = "utf-8-sig" if not file_exists else "utf-8"
            pd.DataFrame([result_dict]).to_csv(
                output_csv, 
                mode="a", 
                header=not file_exists, 
                index=False, 
                encoding=enc,
            )
            file_exists = True
            
            processed.add(task_id)
            count += 1
            time.sleep(1) 

    print("\n🎉 Solder evaluation complete!")

# --- Execution ---
runfolder_evaluation()

### Final Eval

In [ ]:
# ─── Folders to scan for result CSVs ───
RESULTS_FOLDERS = [
    "MathVisionResults", "ChartQAResults", "ScreenQAResults",
    "TurtleBenchResults", "JEEMResults", "PEARLResults",
]

# Add any dataset-specific evaluation column names here.
# The first matching column found in each CSV will be used.
EVAL_COLUMN_CANDIDATES = [
    "evaluation_passed",
    "Evaluation",
    "ComplexEval",
    "LLM_Judge_Result",
    "score",
    "is_perfect",
]


def normalize_column_name(name):
    """Normalize column names so checks are case/spacing/underscore-insensitive."""
    return re.sub(r"[^a-z0-9]", "", str(name).strip().lower())


def find_eval_column(df, candidates=EVAL_COLUMN_CANDIDATES):
    """Return the real CSV column name matching one of the configured candidates."""
    normalized_columns = {normalize_column_name(col): col for col in df.columns}
    for candidate in candidates:
        match = normalized_columns.get(normalize_column_name(candidate))
        if match:
            return match
    return None


def coerce_eval_score(value):
    """Convert bools, numeric scores, and fraction strings into a 0-1 score."""
    if pd.isna(value):
        return 0.0
    if isinstance(value, bool):
        return 1.0 if value else 0.0
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip().lower()
    if text in {"true", "yes", "pass", "passed", "correct", "equivalent"}:
        return 1.0
    if text in {"false", "no", "fail", "failed", "incorrect", "not equivalent", "", "nan", "none"}:
        return 0.0

    fraction_match = re.search(r"(\d+(?:\.\d+)?)\s*(?:/|out\s*of|outof)\s*(\d+(?:\.\d+)?)", text)
    if fraction_match:
        numerator = float(fraction_match.group(1))
        denominator = float(fraction_match.group(2))
        return numerator / denominator if denominator else 0.0

    try:
        return float(text)
    except ValueError:
        return 0.0


def should_show_threshold_accuracy(eval_col):
    """Return True for 0-1 similarity score columns where >= 0.5 is a match."""
    return normalize_column_name(eval_col) in {
        normalize_column_name("Evaluation"),
    }


def summarize_results(folders=RESULTS_FOLDERS):
    """Print mean metric scores and thresholded accuracy where applicable."""
    header = f"{'Model Name':<60} | {'Rows':>8} | {'Metric Column':<18} | {'Mean Score':>10} | {'>=0.5 Accuracy':>14}"
    print(header)
    print("-" * len(header))
    total_rows = 0

    for folder in folders:
        folder_path = Path(folder)
        if not folder_path.exists():
            continue

        print(f"\n{'=' * 24} {folder} {'=' * 24}")
        for csv_file in sorted(folder_path.glob("*.csv")):
            try:
                df = pd.read_csv(csv_file, encoding="utf-8-sig")
                row_count = len(df)
                if row_count == 0:
                    continue

                eval_col = find_eval_column(df)
                if eval_col is None:
                    # print(f"⏭️ Skipping {csv_file.name}: no evaluation column found.")
                    continue

                scores = df[eval_col].apply(coerce_eval_score)
                mean_score = scores.mean() * 100
                threshold_accuracy = (scores >= 0.5).mean() * 100 if should_show_threshold_accuracy(eval_col) else None
                name = csv_file.stem.replace("_openrouter", "").replace("_results", "")
                threshold_display = f"{threshold_accuracy:>12.2f}%" if threshold_accuracy is not None else f"{'N/A':>13}"
                print(f"{name:<60} | {row_count:>8} | {eval_col:<18} | {mean_score:>8.2f}% | {threshold_display}")
                total_rows += row_count
            except Exception as e:
                print(f"⚠️ Error processing {csv_file.name}: {e}")

    print("-" * len(header))
    print(f"{'TOTAL ROWS':<60} | {total_rows:>8}")


summarize_results()

In [78]:
# Count rows in every CSV across all result folders
RESULTS_FOLDERS = [
    "MathVisionResults", 
    "ChartQAResults",
    "ScreenQAResults",
    "TurtleBenchResults", 
    "JEEMResults", 
    "PEARLResults",
]

overall_total_rows = 0

for folder in RESULTS_FOLDERS:
    folder_path = Path(folder)
    if not folder_path.exists():
        print(f"Skipping {folder}: folder not found.")
        continue

    folder_total_rows = 0
    csv_files = sorted(folder_path.glob("*.csv"))

    print(f"\n{'=' * 24} {folder} {'=' * 24}")
    if not csv_files:
        print("No CSV files found.")
        continue

    for csv_file in csv_files:
        try:
            row_count = len(pd.read_csv(csv_file, encoding="utf-8-sig"))
            folder_total_rows += row_count
            overall_total_rows += row_count
            print(f"{csv_file.name:<80} {row_count:>8}")
        except Exception as e:
            print(f"Error reading {csv_file.name}: {e}")

    print(f"{'Folder total':<80} {folder_total_rows:>8}")

print("\n" + "=" * 96)
print(f"{'OVERALL TOTAL ROWS':<80} {overall_total_rows:>8}")



======================== MathVisionResults ========================
mathvision_openrouter_google_gemini-3.1-flash-lite-preview_results.csv               3040
mathvision_openrouter_mistralai_ministral-8b-2512_results.csv                        3040
mathvision_openrouter_openai_gpt-4o-mini_results.csv                                 3040
mathvision_openrouter_openai_gpt-4o-mini_results_GRADED.csv                          3040
mathvision_openrouter_qwen_qwen3-vl-8b-instruct_results.csv                          3040
mathvision_openrouter_qwen_qwen3-vl-8b-instruct_results_GRADED.csv                   3040
mathvision_openrouter_rekaai_reka-edge_results.csv                                   3040
mathvisionarabic_openrouter_google_gemini-3.1-flash-lite-preview_results.csv         3040
mathvisionarabic_openrouter_mistralai_ministral-8b-2512_results.csv                  3040
mathvisionarabic_openrouter_openai_gpt-4o-mini_results.csv                           3040
mathvisionarabic_openrouter_ope

In [82]:
# Compare PEARL parquet row counts with PEARL result CSV row counts by question_type
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

PEARL_PARQUET_PATH = Path("../Datasets/pearl.parquet")
PEARL_RESULTS_DIR = Path("PEARLResults")
QUESTION_TYPE_COLUMN = "question_type"


def normalize_question_type(value):
    if pd.isna(value):
        return "<missing>"
    text = str(value).strip()
    return text if text else "<missing>"


pearl_parquet_file = pq.ParquetFile(PEARL_PARQUET_PATH)
pearl_parquet_rows = pearl_parquet_file.metadata.num_rows
pearl_parquet_qtypes = pd.read_parquet(
    PEARL_PARQUET_PATH, columns=[QUESTION_TYPE_COLUMN]
)[QUESTION_TYPE_COLUMN].apply(normalize_question_type)
parquet_type_counts = pearl_parquet_qtypes.value_counts().sort_index()

print(f"PEARL parquet rows: {pearl_parquet_rows}")
print("\nParquet rows by question_type:")
print(parquet_type_counts.to_string())

summary_rows = []
type_comparison_rows = []

for csv_file in sorted(PEARL_RESULTS_DIR.glob("*.csv")):
    try:
        csv_df = pd.read_csv(csv_file, encoding="utf-8-sig", usecols=lambda col: col == QUESTION_TYPE_COLUMN)
        csv_rows = len(csv_df)
        difference = csv_rows - pearl_parquet_rows

        if QUESTION_TYPE_COLUMN not in csv_df.columns:
            print(f"{csv_file.name}: missing {QUESTION_TYPE_COLUMN} column")
            continue

        csv_type_counts = csv_df[QUESTION_TYPE_COLUMN].apply(normalize_question_type).value_counts().sort_index()
        all_types = sorted(set(parquet_type_counts.index) | set(csv_type_counts.index))

        summary_rows.append({
            "csv_file": csv_file.name,
            "parquet_rows": pearl_parquet_rows,
            "csv_rows": csv_rows,
            "difference_csv_minus_parquet": difference,
            "matches_parquet_total": difference == 0,
        })

        print(f"\n{csv_file.name}")
        print(f"  CSV rows: {csv_rows} | diff vs parquet: {difference}")
        print("  question_type comparison:")

        for qtype in all_types:
            parquet_count = int(parquet_type_counts.get(qtype, 0))
            csv_count = int(csv_type_counts.get(qtype, 0))
            type_diff = csv_count - parquet_count
            type_comparison_rows.append({
                "csv_file": csv_file.name,
                "question_type": qtype,
                "parquet_rows": parquet_count,
                "csv_rows": csv_count,
                "difference_csv_minus_parquet": type_diff,
                "matches_parquet": type_diff == 0,
            })
            print(f"    {qtype:<25} parquet={parquet_count:>6} | csv={csv_count:>6} | diff={type_diff:>6}")

    except Exception as e:
        print(f"Error reading {csv_file.name}: {e}")

pearl_row_count_summary = pd.DataFrame(summary_rows)
pearl_question_type_summary = pd.DataFrame(type_comparison_rows)

pearl_row_count_summary 
# pearl_question_type_summary


PEARL parquet rows: 4832

Parquet rows by question_type:
question_type
multiple_choice    3766
true_false         1066

pearl_openrouter_google_gemini-3.1-flash-lite-preview_results.csv
  CSV rows: 1644 | diff vs parquet: -3188
  question_type comparison:
    multiple_choice           parquet=  3766 | csv=  1267 | diff= -2499
    true_false                parquet=  1066 | csv=   377 | diff=  -689

pearl_openrouter_mistralai_ministral-8b-2512_results.csv
  CSV rows: 1644 | diff vs parquet: -3188
  question_type comparison:
    multiple_choice           parquet=  3766 | csv=  1267 | diff= -2499
    true_false                parquet=  1066 | csv=   377 | diff=  -689

pearl_openrouter_openai_gpt-4o-mini_results.csv
  CSV rows: 1644 | diff vs parquet: -3188
  question_type comparison:
    multiple_choice           parquet=  3766 | csv=  1267 | diff= -2499
    true_false                parquet=  1066 | csv=   377 | diff=  -689

pearl_openrouter_qwen_qwen3-vl-8b-instruct_results.csv
  CSV row

,csv_file,parquet_rows,csv_rows,difference_csv_minus_parquet,matches_parquet_total
0,pearl_openrouter_google_gemini-3.1-flash-lite-...,4832,1644,-3188,False
1,pearl_openrouter_mistralai_ministral-8b-2512_r...,4832,1644,-3188,False
2,pearl_openrouter_openai_gpt-4o-mini_results.csv,4832,1644,-3188,False
3,pearl_openrouter_qwen_qwen3-vl-8b-instruct_res...,4832,1644,-3188,False
4,pearl_openrouter_rekaai_reka-edge_results.csv,4832,1644,-3188,False
